In [1]:
import os
import random
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
from tensorflow.keras.applications.vgg16 import preprocess_input
from tensorflow.keras.applications import VGG16
from tensorflow.keras.metrics import AUC, confusion_matrix, classification_report,
    roc_auc_score, precision_score, recall_score, f1_score, roc_curve, auc
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import BinaryFocalCrossentropy
import time
import csv
from datetime import datetime
import matplotlib.pyplot as plt

# Mixed precision
tf.keras.mixed_precision.set_global_policy("mixed_float16")

print("Num GPUs Available:", len(tf.config.list_physical_devices("GPU")))
print("Mixed precision policy:", tf.keras.mixed_precision.global_policy())

CLASS_NAMES = ["Normal", "Abnormal"]

SEED = 0
tf.random.set_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

CSV_FILE = "vgg_experiments_log.csv"
EXPERIMENTS_FOLDER = "experiments_VGG16"
CURVE_PLOT_FOLDER = os.path.join(EXPERIMENTS_FOLDER, "loss_plots")
ROC_PLOT_FOLDER = os.path.join(EXPERIMENTS_FOLDER, "roc_plots")

# Ensure folders exist
os.makedirs(CURVE_PLOT_FOLDER, exist_ok=True)
os.makedirs(ROC_PLOT_FOLDER, exist_ok=True)
os.makedirs(EXPERIMENTS_FOLDER, exist_ok=True)

# Configurable parameters
IMG_SIZE = 299        # changeable
CHANNELS = 3
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

# Training parameters
EPOCHS = 30
EARLY_STOP_PATIENCE = 5
BASE_LR = 1e-5

# Callbacks
lr_reduce = ReduceLROnPlateau(
    monitor="val_roc_auc",
    factor=0.2,
    patience=3,
    min_lr=1e-6,
    mode="max",
    verbose=1
)

early_stop = EarlyStopping(
    monitor="val_roc_auc",
    patience=EARLY_STOP_PATIENCE,
    mode="max",
    restore_best_weights=True,
    verbose=1
)

# Data paths
TRAIN_IMAGES_PATH = "images_bal_bin.npy"
TRAIN_LABELS_PATH = "labels_bal_bin.npy"

# VAL_IMAGES_PATH = "Kaggle_Data_recheck/Cleaned_Validation_Data/cv10_data.npy"
# VAL_LABELS_PATH = "Kaggle_Data_recheck/Cleaned_Validation_Data/cv10_labels.npy"

VAL_IMAGES_PATH = "Kaggle_Data_recheck/Cleaned_Test_Data/test10_data.npy"
VAL_LABELS_PATH = "Kaggle_Data_recheck/Cleaned_Test_Data/test10_labels.npy"

print("IMG_SIZE:", IMG_SIZE)
print("BATCH_SIZE:", BATCH_SIZE)
print("Epochs:", EPOCHS)

2026-03-11 22:53:12.266060: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-11 22:53:12.283396: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-03-11 22:53:12.305444: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8473] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-03-11 22:53:12.312595: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1471] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-03-11 22:53:12.328363: I tensorflow/core/platform/cpu_feature_guar

INFO:tensorflow:Mixed precision compatibility check (mixed_float16): OK
Your GPU will likely run quickly with dtype policy mixed_float16 as it has compute capability of at least 7.0. Your GPU: NVIDIA GeForce RTX 5070 Ti Laptop GPU, compute capability 12.0
Num GPUs Available: 1
Mixed precision policy: <Policy "mixed_float16">
IMG_SIZE: 299
BATCH_SIZE: 32
Epochs: 30


I0000 00:00:1773269594.560182    5411 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1773269594.622844    5411 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1773269594.624598    5411 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1773269594.626419    5411 cuda_executor.cc:1015] successful NUMA node read from SysFS ha

In [2]:
# Load numpy arrays
X_train = np.load(TRAIN_IMAGES_PATH)
y_train = np.load(TRAIN_LABELS_PATH)

X_val = np.load(VAL_IMAGES_PATH)
y_val = np.load(VAL_LABELS_PATH)

# Relabel: 0:0, 1:1, 2:1
y_train = (y_train > 0).astype(np.float32)
y_val   = (y_val > 0).astype(np.float32)

print("Train shape:", X_train.shape, y_train.shape)
print("Val shape:", X_val.shape, y_val.shape)
print("Train class balance:", np.mean(y_train))
print("Val class balance:", np.mean(y_val))

# Preprocessing function
def preprocess_image(img, label, augment=False):
    img = tf.cast(img, tf.float32)

    # Ensure channel dimensions
    if img.shape.rank == 2:
        img = tf.expand_dims(img, axis=-1)

    # Grayscale to RGB
    img = tf.image.grayscale_to_rgb(img)

    # Resize
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))

    if augment:
        img = tf.image.random_flip_left_right(img)
        img = tf.image.random_brightness(img, 0.1)
        img = tf.image.random_contrast(img, 0.9, 1.1)

    img = preprocess_input(img)

    return img, label

# tf.data pipelines
train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train))
train_ds = (
    train_ds
    .shuffle(2048, seed=SEED)
    .map(lambda x, y: preprocess_image(x, y, augment=True), num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

val_ds = tf.data.Dataset.from_tensor_slices((X_val, y_val))
val_ds = (
    val_ds
    .map(lambda x, y: preprocess_image(x, y, augment=False), num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

print("Datasets ready (binary labels)")

Train shape: (14578, 299, 299, 1) (14578,)
Val shape: (7682, 299, 299, 1) (7682,)
Train class balance: 0.5
Val class balance: 0.12822181


I0000 00:00:1773269596.731399    5411 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1773269596.734073    5411 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1773269596.736439    5411 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1773269596.866279    5411 cuda_executor.cc:1015] successful NUMA node read from SysFS ha

Datasets ready (binary labels)


In [3]:
# Check one batch
for images, labels in train_ds.take(1):
    print("Train batch images shape:", images.shape)
    print("Train batch labels shape:", labels.shape)
    print("Image dtype:", images.dtype, "Label dtype:", labels.dtype)
    print("Image min/max:", tf.reduce_min(images).numpy(), tf.reduce_max(images).numpy())

for images, labels in val_ds.take(1):
    print("Validation batch images shape:", images.shape)
    print("Validation batch labels shape:", labels.shape)
    print("Image dtype:", images.dtype, "Label dtype:", labels.dtype)
    print("Image min/max:", tf.reduce_min(images).numpy(), tf.reduce_max(images).numpy())

Train batch images shape: (32, 299, 299, 3)
Train batch labels shape: (32,)
Image dtype: <dtype: 'float32'> Label dtype: <dtype: 'float32'>
Image min/max: -123.701004 -102.974
Validation batch images shape: (32, 299, 299, 3)
Validation batch labels shape: (32,)
Image dtype: <dtype: 'float32'> Label dtype: <dtype: 'float32'>
Image min/max: -123.68 122.061


2026-03-11 22:53:27.685415: I tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2026-03-11 22:53:27.794893: I tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [4]:
# Backbone
base_model = VGG16(
    weights="imagenet",
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, CHANNELS)
)

base_model.trainable = False

# Classification head
inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, CHANNELS))
x = base_model(inputs, training=True)

x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)

x = layers.Dense(512, activation="relu")(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.5)(x)

x = layers.Dense(256, activation="relu")(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.5)(x)

outputs = layers.Dense(1, activation="sigmoid", dtype="float32")(x)

model = keras.Model(inputs, outputs)

# # Compile
# model.compile(
#     optimizer=Adam(learning_rate=BASE_LR),
#     loss=BinaryFocalCrossentropy(gamma=2.0, alpha=0.25),
#     metrics=[
#         AUC(name="roc_auc", curve="ROC")
#     ]
# )

# model.summary()

In [5]:
# Freeze everything
for layer in base_model.layers:
    layer.trainable = False

# Unfreeze convolutional layers block by block
for layer in base_model.layers:
    if layer.name.startswith("block5_conv"):
        layer.trainable = True
    if layer.name.startswith("block4_conv"):
        layer.trainable = True
    # if layer.name.startswith("block3_conv"):
    #     layer.trainable = True
    # if layer.name.startswith("block2_conv"):
    #     layer.trainable = True
    # # Freeze BatchNorm layers
    # if isinstance(layer, tf.keras.layers.BatchNormalization):
    #     layer.trainable = False

# Verification
print("Trainable layers:")
for layer in base_model.layers:
    if layer.trainable:
        print("  ", layer.name)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=BASE_LR),
    loss=tf.keras.losses.BinaryFocalCrossentropy(gamma=3.0, alpha=0.5),
    metrics=[
        tf.keras.metrics.AUC(name="roc_auc")
    ]
)

model.summary()

Trainable layers:
   block4_conv1
   block4_conv2
   block4_conv3
   block5_conv1
   block5_conv2
   block5_conv3
Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_2 (InputLayer)        [(None, 299, 299, 3)]     0         
                                                                 
 vgg16 (Functional)          (None, 9, 9, 512)         14714688  
                                                                 
 global_average_pooling2d (  (None, 512)               0         
 GlobalAveragePooling2D)                                         
                                                                 
 batch_normalization (Batch  (None, 512)               2048      
 Normalization)                                                  
                                                                 
 dense (Dense)               (None, 512)               262656    
             

In [6]:
# Callbacks
callbacks = [early_stop, lr_reduce]

# Time model training
start_time = time.time()

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

training_time_sec = time.time() - start_time
print(f"Training time (min): {training_time_sec / 60:.2f}")

Epoch 1/30


2026-03-11 22:53:33.884850: E tensorflow/core/util/util.cc:131] oneDNN supports DT_HALF only on platforms with AVX-512. Falling back to the default Eigen-based implementation if present.
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
2026-03-11 22:53:34.084521: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:531] Loaded cuDNN version 90700
W0000 00:00:1773269614.181248    5513 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1773269614.182906    5513 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1773269614.184616    5513 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1773269614.196464    5513 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced

  1/456 [..............................] - ETA: 1:09:32 - loss: 0.5041 - roc_auc: 0.5142

'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
W0000 00:00:1773269620.023056    5500 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1773269620.024294    5500 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1773269620.025376    5500 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1773269620.027664    5500 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1773269620.029644    5500 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1773269620.032041    5500 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1773269620.034479    5500 gpu_timer.cc:114] Skipping

  2/456 [..............................] - ETA: 12:58 - loss: 0.3976 - roc_auc: 0.5873  

...

455/456 [============================>.] - ETA: 0s - loss: 0.3874 - roc_auc: 0.6457  

W0000 00:00:1773269653.246990    5517 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1773269653.247595    5517 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1773269653.248151    5517 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1773269653.249674    5517 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1773269653.251031    5517 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1773269653.252582    5517 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1773269653.253992    5517 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1773269653.256405    5517 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1773269653.258581    5517 gp

456/456 [==============================] - ETA: 0s - loss: 0.3875 - roc_auc: 0.6460

W0000 00:00:1773269654.057618    5517 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1773269654.060085    5517 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1773269654.062559    5517 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1773269654.064964    5517 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1773269654.067586    5517 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1773269654.070158    5517 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1773269654.072709    5517 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1773269654.075345    5517 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1773269654.078071    5517 gp

456/456 [==============================] - 59s 109ms/step - loss: 0.3875 - roc_auc: 0.6460 - val_loss: 29.7708 - val_roc_auc: 0.5944 - lr: 1.0000e-05
Epoch 2/30
456/456 [==============================] - 44s 96ms/step - loss: 0.3359 - roc_auc: 0.7014 - val_loss: 264.4512 - val_roc_auc: 0.5807 - lr: 1.0000e-05
Epoch 3/30
456/456 [==============================] - 44s 96ms/step - loss: 0.3274 - roc_auc: 0.7038 - val_loss: 255.6517 - val_roc_auc: 0.5885 - lr: 1.0000e-05
Epoch 4/30
455/456 [============================>.] - ETA: 0s - loss: 0.3161 - roc_auc: 0.7067  
Epoch 4: ReduceLROnPlateau reducing learning rate to 1.9999999494757505e-06.
456/456 [==============================] - 44s 96ms/step - loss: 0.3160 - roc_auc: 0.7067 - val_loss: 274.9208 - val_roc_auc: 0.5843 - lr: 1.0000e-05
Epoch 5/30
456/456 [==============================] - 44s 96ms/step - loss: 0.3172 - roc_auc: 0.7135 - val_loss: 286.2871 - val_roc_auc: 0.5855 - lr: 2.0000e-06
Epoch 6/30
456/456 [=======================

In [7]:
# Validationpredictions
val_ds_eval = val_ds.unbatch().batch(32)
y_val_bin = np.concatenate([y for x, y in val_ds], axis=0)

y_pred_probs = model.predict(val_ds_eval).ravel()

# Default threshold = 0.5
y_pred = (y_pred_probs >= 0.5).astype(int)

# ROC-AUC
roc_auc = roc_auc_score(y_val_bin, y_pred_probs)
print(f"\nROC-AUC: {roc_auc:.4f}")

# Confusion matrix
cm = confusion_matrix(y_val_bin, y_pred)
cm_norm = cm.astype("float") / cm.sum(axis=1, keepdims=True)
print("\nConfusion matrix:\n", cm)
print("\nNormalized confusion matrix:\n", np.round(cm_norm, 2))

# Classification report
report_dict = classification_report(
    y_val_bin,
    y_pred,
    target_names=CLASS_NAMES,
    output_dict=True
)
print("\nClassification report:\n", classification_report(y_val_bin, y_pred, target_names=CLASS_NAMES))

# Per-class accuracy
per_class_acc = cm.diagonal() / cm.sum(axis=1)
for i, acc in enumerate(per_class_acc):
    print(f"Accuracy for class {i} ({CLASS_NAMES[i]}): {acc:.4f}")


# Threshold analysis
thresholds = np.linspace(0.0, 1.0, 201)  # step=0.005
rows = []

for t in thresholds:
    y_pred_t = (y_pred_probs >= t).astype(int)
    cm_t = confusion_matrix(y_val_bin, y_pred_t)
    
    # Skip degenerate thresholds
    if cm_t.shape != (2, 2):
        continue

    tn, fp, fn, tp = cm_t.ravel()
    
    # Per-class accuracy
    acc_normal = tn / (tn + fp) if (tn + fp) > 0 else 0
    acc_abnormal = tp / (tp + fn) if (tp + fn) > 0 else 0

    # Overall metrics
    accuracy = (tp + tn) / (tp + tn + fp + fn)
    precision_abn = precision_score(y_val_bin, y_pred_t, pos_label=1, zero_division=0)
    recall_abn = recall_score(y_val_bin, y_pred_t, pos_label=1, zero_division=0)
    f1_abn = f1_score(y_val_bin, y_pred_t, pos_label=1, zero_division=0)

    rows.append({
        "threshold": t,
        "accuracy_overall": accuracy,
        "acc_normal": acc_normal,
        "acc_abnormal": acc_abnormal,
        "precision_abnormal": precision_abn,
        "recall_abnormal": recall_abn,
        "f1_abnormal": f1_abn,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
    })

df_thresholds = pd.DataFrame(rows)

# Apply example constraints
filtered = df_thresholds[
    (df_thresholds["acc_normal"] >= 0.52) &
    (df_thresholds["acc_abnormal"] >= 0.64)
].copy()

print(f"\nThresholds satisfying constraints: {len(filtered)}")

# Sort by metrics of interest
filtered = filtered.sort_values(by=["recall_abnormal", "f1_abnormal"], ascending=False)

pd.set_option("display.float_format", "{:.3f}".format)
print("\nTop thresholds:\n", filtered.head(10))

241/241 [==============================] - 15s 61ms/step

ROC-AUC: 0.6150

Confusion matrix:
 [[2067 4630]
 [ 138  847]]

Normalized confusion matrix:
 [[0.31 0.69]
 [0.14 0.86]]

Classification report:
               precision    recall  f1-score   support

      Normal       0.94      0.31      0.46      6697
    Abnormal       0.15      0.86      0.26       985

    accuracy                           0.38      7682
   macro avg       0.55      0.58      0.36      7682
weighted avg       0.84      0.38      0.44      7682

Accuracy for class 0 (Normal): 0.3086
Accuracy for class 1 (Abnormal): 0.8599


2026-03-11 22:58:26.283809: I tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
2026-03-11 22:58:26.283859: I tensorflow/core/framework/local_rendezvous.cc:423] Local rendezvous recv item cancelled. Key hash: 8290195979019953318



Thresholds satisfying constraints: 0

Top thresholds:
 Empty DataFrame
Columns: [threshold, accuracy_overall, acc_normal, acc_abnormal, precision_abnormal, recall_abnormal, f1_abnormal, tn, fp, fn, tp]
Index: []


In [8]:
# Auto-increment experiment ID
def get_next_experiment_id(csv_file):
    if not os.path.exists(csv_file):
        return 1
    with open(csv_file, "r") as f:
        return sum(1 for _ in f)

# Extract max L1/L2 across layers
def extract_l1_l2(model):
    l1_vals, l2_vals = [], []
    for layer in model.layers:
        reg = getattr(layer, "kernel_regularizer", None)
        if reg:
            if hasattr(reg, "l1") and reg.l1 is not None:
                l1_vals.append(float(tf.keras.backend.get_value(reg.l1)))
            if hasattr(reg, "l2") and reg.l2 is not None:
                l2_vals.append(float(tf.keras.backend.get_value(reg.l2)))
    return (
        max(l1_vals) if l1_vals else 0.0,
        max(l2_vals) if l2_vals else 0.0)

# Extract max dropout rate
def extract_max_dropout(model):
    rates = []
    for layer in model.layers:
        if isinstance(layer, tf.keras.layers.Dropout):
            rates.append(float(layer.rate))
    return max(rates) if rates else 0.0

# Build experiment row
exp_id = get_next_experiment_id(CSV_FILE)
l1_val, l2_val = extract_l1_l2(model)
dropout_val = extract_max_dropout(model)
learning_rate_val = float(model.optimizer.learning_rate.numpy())
train_loss = history.history["loss"][-1]
val_loss = history.history["val_loss"][-1]

row = {
    "experiment_id": exp_id,
    "timestamp": datetime.now().isoformat(),
    "seed": SEED,
    "task": "binary roc-auc",
    # "backbone": BACKBONE_NAME,
    # "pretrained": PRETRAINED,
    "input_size": IMG_SIZE,
    "batch_size": BATCH_SIZE,
    # "shuffle_buffer": SHUFFLE_BUFFER,
    "epochs": len(history.history["loss"]),
    "learning_rate": learning_rate_val,
    "dropout_rate": dropout_val,
    "l1": l1_val,
    "l2": l2_val,
    "train_loss": train_loss,
    "val_loss": val_loss,
    "training_time_min": training_time_sec / 60.0,
    "roc_auc": roc_auc,  # from report cell
    "num_thresholds_meeting_constraints": len(filtered)
}

# Overall accuracy
overall_acc = np.mean((y_pred_probs >= 0.5).astype(int) == y_val_bin)
row["accuracy"] = overall_acc

# Per-class metrics
for i, cls in enumerate(CLASS_NAMES):
    cls_l = cls.lower()
    row[f"acc_{cls_l}"] = per_class_acc[i]
    row[f"precision_{cls_l}"] = report_dict[cls]["precision"]
    row[f"recall_{cls_l}"] = report_dict[cls]["recall"]
    row[f"f1_{cls_l}"] = report_dict[cls]["f1-score"]

# Save to CSV
file_exists = os.path.isfile(CSV_FILE)
with open(CSV_FILE, "a", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=row.keys())
    if not file_exists:
        writer.writeheader()
    writer.writerow(row)

print(f"Experiment {exp_id} logged to {CSV_FILE}")

# Save training curves
plt.figure(figsize=(8,6))
plt.plot(history.history["loss"], label="Train Loss", marker='o')
plt.plot(history.history["val_loss"], label="Val Loss", marker='o')
if "accuracy" in history.history:
    plt.plot(history.history["accuracy"], label="Train Acc", marker="x")
    plt.plot(history.history["val_accuracy"], label="Val Acc", marker="x")
plt.title(f"Training Curves - Experiment {exp_id:05d}")
plt.xlabel("Epoch")
plt.ylabel("Value")
plt.legend()
plt.grid(True)
plt.savefig(os.path.join(CURVE_PLOT_FOLDER, f"{exp_id:05d}.png"))
plt.close()

# Save ROC curve
fpr, tpr, thresholds_roc = roc_curve(y_val_bin, y_pred_probs)
roc_auc_val = auc(fpr, tpr)

plt.figure(figsize=(6,6))
plt.plot(fpr, tpr, color="blue", lw=2, label=f"ROC curve (AUC = {roc_auc_val:.3f})")
plt.plot([0,1], [0,1], color="gray", lw=1, linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title(f"ROC Curve - Experiment {exp_id:05d}")
plt.legend(loc="lower right")
plt.grid(True)
plt.savefig(os.path.join(ROC_PLOT_FOLDER, f"{exp_id:05d}.png"))
plt.close()
print(f"ROC curve saved to {ROC_PLOT_FOLDER}")


Experiment 27 logged to vgg_experiments_log.csv
ROC curve saved to experiments_VGG16/roc_plots
